In [1]:
import os
import time
from matplotlib import pyplot as plt
import seaborn as sns
from scr.qsvdd_core.data_loader import QuantumDataLoader
import numpy as np
import tensorflow as tf
import pennylane as qml
import pandas as pd
from scipy.io import arff
from ucimlrepo import fetch_ucirepo
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from scripts.train_model import circuit_training, train_five_times
from scr.qsvdd_core.data_loader import QuantumDataLoader
from scripts.test_model import test, mean_auc, best_batch, save_test_results

2026-05-14 14:51:04.755901: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-14 14:51:04.756664: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-14 14:51:04.863083: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
np.random.seed(42)

In [14]:
n_train = 0 ; latent_dim = 3
# num_params_conv = 375
cost_func = 'svdd'

# Breast Cancer anomaly Detection

In [38]:
print("="*60)
print("Loading and processing the Breast Cancer dataset...")
print("="*60)

# 1. Fetch the dataset
breast_cancer = fetch_ucirepo(id=17)
X_bc_raw = breast_cancer.data.features
y_bc_raw = breast_cancer.data.targets

Loading and processing the Breast Cancer dataset...


In [ ]:
print(breast_cancer.data.features.head())

In [ ]:
X_bc_raw.info()

In [ ]:
y_bc_raw.info()

In [27]:
y_bc = y_bc_raw.iloc[:, 0].apply(lambda x: 1 if x == 'M' else 0).values

loader = QuantumDataLoader()

import pandas as pd
df_bc = pd.DataFrame(X_bc_raw)
df_bc['Class'] = y_bc

X_data = loader.prepare_bc_data(X_bc_raw)
y = y_bc

# 4. Identify indices for each class
normal_indices = np.where(y == 0)[0]
abnormal_indices = np.where(y == 1)[0]
np.random.seed(42)

# 5. Training Set (One-Class: 200 normal samples)
X_train_normal_indices = np.random.choice(normal_indices, 250, replace=False)
X_train = X_data[X_train_normal_indices]
Y_train = y[X_train_normal_indices]

# 6. Test Set (Balanced: 50 normal + 50 anomalies)
remaining_normal_indices = list(set(normal_indices) - set(X_train_normal_indices))

X_test_normal_indices = np.random.choice(remaining_normal_indices, 50, replace=False)
X_test_abnormal_indices = np.random.choice(abnormal_indices, 50, replace=False)

X_test_normal = X_data[X_test_normal_indices]
y_test_normal = y[X_test_normal_indices]

X_test_abnormal = X_data[X_test_abnormal_indices]
y_test_abnormal = y[X_test_abnormal_indices]

# Final test set (Mix: 100 samples)
X_test = np.concatenate((X_test_normal, X_test_abnormal), axis=0)
Y_test = np.concatenate((y_test_normal, y_test_abnormal), axis=0)

print(f"X_train shape (Normal): {X_train.shape}")
print(f"Y_train shape: {Y_train.shape}")
print(f"X_test shape (Mix): {X_test.shape}")
print(f"Y_test shape: {Y_test.shape}")

X_train shape (Normal): (250, 32)
Y_train shape: (250,)
X_test shape (Mix): (100, 32)
Y_test shape: (100,)


In [28]:
loader = QuantumDataLoader()

X_quantum = loader.prepare_bc_data(X_bc_raw)

print(f"Features for the circuit: {X_quantum.shape}")
print(f"Labels: {y.shape}")

Features for the circuit: (569, 32)
Labels: (569,)


# ALOI Dataset

In [14]:
data, meta = arff.loadarff('../data/ALOI_withoutdupl_norm.arff')
df = pd.DataFrame(data)

# Pegamos da primeira coluna até a 'att27' (índice 0 até 26)
# O fatiamento :27 pega os índices de 0 a 26.
X = df.iloc[:, :27].values

# 3. Tratar o Label (y)
# A coluna de label chama-se 'outlier'
y_raw = df['outlier']

# Converter bytes para string e depois para binário (yes=1, no=0)
y = y_raw.apply(lambda x: x.decode('utf-8').lower() if isinstance(x, bytes) else str(x).lower())
y = np.where(y == 'yes', 1, 0)

# 4. Resultados Finais
print(f"{'='*30}")
print(f"DATASET ALOI CARREGADO")
print(f"{'='*30}")
print(f"Instâncias: {X.shape[0]}")
print(f"Atributos (Features): {X.shape[1]} (att1 até att27)")
print(f"Anomalias (Outliers): {np.sum(y)}")
print(f"Proporção de Outliers: {np.mean(y)*100:.2f}%")
print(f"{'='*30}")

DATASET ALOI CARREGADO
Instâncias: 49534
Atributos (Features): 27 (att1 até att27)
Anomalias (Outliers): 1508
Proporção de Outliers: 3.04%


In [17]:
def prepare_aloi_classic(df):
    # No ALOI, as colunas são 'att1'...'att27', o label é 'outlier' e tem o 'id'
    # Selecionamos as 27 colunas de atributos
    X = df.iloc[:, :27].values

    # Tratamos o label 'outlier' para 0 e 1
    y_raw = df['outlier'].apply(lambda x: x.decode('utf-8').lower() if isinstance(x, bytes) else str(x).lower())
    y = np.where(y_raw == 'yes', 1, 0)

    # Aplicar o Scaler (opcional para ALOI, mas mantém consistência com seu projeto)
    scaler = StandardScaler()
    X_classic = scaler.fit_transform(X)

    X_padded = np.pad(X_classic, ((0, 0), (0, 5)), mode='constant', constant_values=0)
    print(f"Shape para o QSVDD: {X_padded.shape}") # (49534, 32)
    # troca x_classic por x_padded no return se for quantum_algorithm
    return X_padded, y


# Executando a preparação
X_quantum, y = prepare_aloi_classic(df)

print(f"Features para o modelo: {X_data.shape}")
print(f"Labels: {y.shape}")
print(f"Total de Anomalias: {np.sum(y)}")

Shape para o QSVDD: (49534, 32)
Features para o modelo: (49534, 32)
Labels: (49534,)
Total de Anomalias: 1508


# Credit Card

In [39]:
file_path = os.path.join("..", "data", "creditcard.csv")
df = pd.read_csv(file_path)

print("Dataset loaded successfully!\n")
print("First 5 records:\n", df.head())

Dataset loaded successfully!

First 5 records:
    Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.14126

In [40]:
tmp = df[['Amount','Class']].copy()
class_0 = tmp.loc[tmp['Class'] == 0]['Amount']
class_1 = tmp.loc[tmp['Class'] == 1]['Amount']

In [41]:
loader = QuantumDataLoader()

X_quantum, y = loader.prepare_fraud_data(df)

print(f"Features for the circuit: {X_quantum.shape}")
print(f"Labels: {y.shape}")

Features for the circuit: (284807, 32)
Labels: (284807,)


# MNIST

In [3]:
# Declare the normal class and the dimension of the latent space
# For QSVDD: num_params_conv <- 15, cost_func <- 'svdd', steps = 500
# For QAE: num_params_conv <- 8, cost_func <- 'qae', steps = 2000

# dataset <- 'mnist', 'fmnist', 'cifar'
# ntrain <- number of class want to train
# latent_dim <- 3, 6, 9, 12, 15

dataset = 'mnist'
ntrain = 0 ; latent_dim = 3
num_params_conv = 15
cost_func = 'svdd'

In [4]:
# Import data

def data(ntrain, latent_dim, dataset):
    if dataset == 'mnist':
        (x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
    elif dataset == 'fmnist':
        (x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()


    x_train, x_test = x_train[..., np.newaxis] / 255.0, x_test[..., np.newaxis] / 255.0  # normalize the data

    x_train_filter = np.where(y_train == ntrain)
    x_test_filter = np.where(y_test == ntrain)

    X_train = x_train[x_train_filter]
    X_test = x_test[x_test_filter]
    Y_train = y_train[x_train_filter]
    Y_test = y_test[x_test_filter]


    train_indices = np.random.choice(len(X_train), 600, replace=False)
    test_indices = np.random.choice(len(X_test), 100, replace=False)

    X_train = X_train[train_indices]
    X_test = X_test[test_indices]
    Y_train = Y_train[train_indices]
    Y_test = Y_test[test_indices]

    X_train = tf.image.resize(X_train[:], (256, 1)).numpy()
    X_test = tf.image.resize(X_test[:], (256, 1)).numpy()
    X_train, X_test = tf.squeeze(X_train).numpy(), tf.squeeze(X_test).numpy()

    x_train = tf.image.resize(x_train[:], (256, 1)).numpy()
    x_test = tf.image.resize(x_test[:], (256, 1)).numpy()
    x_train, x_test = tf.squeeze(x_train).numpy(), tf.squeeze(x_test).numpy()

    center = qml.numpy.zeros(latent_dim, requires_grad=True)
    center_train = np.tile(center,(len(X_train),1))
    print('x_train:',x_train.shape)
    print('x_test:',x_test.shape)
    print('X_train:',X_train.shape)
    print('X_test:',X_test.shape)
    print('Y_train:',Y_train.shape)
    print('Y_test:',Y_test.shape)
    return x_train, y_train, x_test, y_test, X_train, X_test, Y_train, Y_test, center_train

In [5]:
x_train, y_train, x_test, y_test, X_train, X_test, Y_train, Y_test, center_train = data(ntrain, latent_dim, dataset)

E0000 00:00:1778781083.177386  778446 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1778781083.193892  778446 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


x_train: (60000, 256)
x_test: (10000, 256)
X_train: (600, 256)
X_test: (100, 256)
Y_train: (600,)
Y_test: (100,)


# Preparing Quantum Data

In [ ]:
"""
One-class Training:
Separation into normal and fraudulent examples
QSVDD will learn what is normal.
"""
normal_indices = np.where(y == 0)[0]
abnormal_indices = np.where(y == 1)[0]
np.random.seed(42)

# Train
# collect 1000 normal examples for training
X_train_normal_indices = np.random.choice(normal_indices, 1000, replace=False)
X_train_normal = X_quantum[X_train_normal_indices]
y_train_normal = y[X_train_normal_indices]

# X_train contains only legitimate transactions
X_train = X_train_normal
Y_train = y_train_normal

# Test (balanced)
# The code removes 100 normal examples that were not used in training
remaining_normal_indices = list(set(normal_indices) - set(X_train_normal_indices))
X_test_normal_indices = np.random.choice(remaining_normal_indices, 100, replace=False)
X_test_normal = X_quantum[X_test_normal_indices]
y_test_normal = y[X_test_normal_indices]

# The code removes 100 fraud examples.
X_test_abnormal_indices = np.random.choice(abnormal_indices, 100, replace=False)
X_test_abnormal = X_quantum[X_test_abnormal_indices]
y_test_abnormal = y[X_test_abnormal_indices]

# creates a test set with 200 examples (50% normal, 50% fraud)
X_test = np.concatenate((X_test_normal, X_test_abnormal), axis=0)
Y_test = np.concatenate((y_test_normal, y_test_abnormal), axis=0)

center = np.zeros(latent_dim)
center_train = np.tile(center, (len(X_train), 1))

print(f'X_train shape: {X_train.shape}')
print(f'Y_train shape: {Y_train.shape}')
print(f'X_test_normal shape: {X_test_normal.shape}')
print(f'X_test_abnormal shape: {X_test_abnormal.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'Y_test shape: {Y_test.shape}')
print(f'center_train shape: {center_train.shape}')

In [6]:
train_Xdata = X_train
train_Ydata = center_train

### QCNN (Quantum Convolutional Neural Network) Ansatz

#### Hyperparameters

In [7]:
qcnn_batch_size = 4
qcnn_steps = 10
qcnn_learning_rate = 0.01

In [20]:
(qcnn_loss_history_matrix,
 qcnn_est_params_matrix,
 qcnn_param_history_matrix,
 qcnn_time_record) = train_five_times(X_train=train_Xdata,
                                        Y_train=train_Ydata,
                                        batch_size=qcnn_batch_size,
                                        learning_rate=qcnn_learning_rate,
                                        steps=qcnn_steps,
                                        ansatz='qcnn'
                                        )
loss_history_f_name = f"../results/training/QCNN/MISC_QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}LR{qcnn_learning_rate:.0e}_LOSS_HISTORY_MEAN.npy"
est_params_f_name = f"../results/training/QCNN/MISC_QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}LR{qcnn_learning_rate:.0e}_EST_PARAMS_MEAN.npy"
time_f_name = f"../results/training/QCNN/MISC_QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}LR{qcnn_learning_rate:.0e}_TIME_MEAN.npy"
np.savetxt(loss_history_f_name, qcnn_loss_history_matrix)
np.savetxt(est_params_f_name, qcnn_est_params_matrix)
np.savetxt(time_f_name, qcnn_time_record)
print("--- All training batches completed ---")

--- Starting training round 1 with seed 1618023241 ---


/home/jvfg/Documents/ORG/Repos/QSVDD2/.venv/lib/python3.12/site-packages/autograd/numpy/numpy_vjps.py:943: ComplexWarning: Casting complex values to real discards the imaginary part
  onp.add.at(A, idx, x)


--- Starting training round 2 with seed 2007863365 ---
--- Starting training round 3 with seed 2587409122 ---
--- Starting training round 4 with seed 1430560531 ---
--- Starting training round 5 with seed 3426467898 ---
--- All training batches completed ---


#### QCNN Training Evaluation

In [32]:
f_name = f"../results/training/QCNN/MISC_QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}LR{qcnn_learning_rate:.0e}_EST_PARAMS_MEAN.npy"
params_list = np.loadtxt(f_name)
mean, std = mean_auc(params_list, n_train, X_test, Y_test, center_train, noisy=False, ansatz="qcnn")

print(50*"--")
print(f'for: B{qcnn_batch_size}S{qcnn_steps} | AUC_mean: {mean} | std: {std}')
print(50*"--")

Processing class 0 (label 0) | Samples: 100
Finished class 0 in 1.68s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 1.67s
Test completed in 3.37s | AUC: 0.4694
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 1.82s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 1.69s
Test completed in 3.51s | AUC: 0.5167
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 1.81s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 1.67s
Test completed in 3.48s | AUC: 0.4742
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 1.84s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 1.66s
Test completed in 3.50s | AUC: 0.4583
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 1.66s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 1.82s
Test completed in 3.48s | AUC: 0.4720
----------------------------------------------------------------------------------------------------
for: B4S2

### QAE (Quantum AutoEncoder) Ansatz

In [10]:
qae_batch_size = 16
qae_steps = 500
qae_learning_rate = 0.001

#### Five-Run Training & Result Persistence

Run `train_five_times` to train the **QAE** ansatz across 5 independent runs with the configured batch size and step count.

In [9]:
(qae_loss_history_matrix,
 qae_est_params_matrix,
 qae_param_history_matrix,
 qae_time_record) = train_five_times(X_train=train_Xdata,
                                        Y_train=train_Ydata,
                                        batch_size=qae_batch_size,
                                        learning_rate=qae_learning_rate,
                                        steps=qae_steps,
                                        ansatz='qae'
                                        )
loss_history_f_name = f"../results/training/QAE/MISC_QAE_B{qae_batch_size:02d}S{qae_steps}LR{qae_learning_rate:.0e}_LOSS_HISTORY_MEAN.npy"
est_params_f_name = f"../results/training/QAE/MISC_QAE_B{qae_batch_size:02d}S{qae_steps}LR{qae_learning_rate:.0e}_EST_PARAMS_MEAN.npy"
time_f_name = f"../results/training/QAE/MISC_QAE_B{qae_batch_size:02d}S{qae_steps}LR{qae_learning_rate:.0e}_TIME_MEAN.npy"
np.savetxt(loss_history_f_name, qae_loss_history_matrix)
np.savetxt(est_params_f_name, qae_est_params_matrix)
np.savetxt(time_f_name, qae_time_record)
print("--- All training batches completed ---")

--- Starting training round 1 with seed 151105533 ---
--- Starting training round 2 with seed 3219205416 ---


KeyboardInterrupt: 

#### QAE Training Evaluation

In [43]:
f_name = f"../results/training/QAE/MISC_QAE_B{qae_batch_size:02d}S{qae_steps}LR{qae_learning_rate:.0e}_EST_PARAMS_MEAN.npy"
params_list = np.loadtxt(f_name)
mean, std = mean_auc(params_list, n_train, X_test, Y_test, center_train, noisy=False, ansatz="qae")

print(50*"--")
print(f'for: B{qae_batch_size}S{qae_steps} | AUC_mean: {mean} | std: {std}')
print(50*"--")

Processing class 0 (label 0) | Samples: 100
Finished class 0 in 1.18s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 1.14s
Test completed in 2.33s | AUC: 0.6583
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 1.14s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 1.14s
Test completed in 2.28s | AUC: 0.7904
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 1.14s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 1.15s
Test completed in 2.29s | AUC: 0.8496
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 1.16s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 1.15s
Test completed in 2.31s | AUC: 0.3881
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 1.16s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 1.16s
Test completed in 2.33s | AUC: 0.7812
----------------------------------------------------------------------------------------------------
for: B16S

### LCQHNN (Lean classical-quantum hybrid neural network) Ansatz

The latent space for this ansatz has dimension 5 (vs. 3 for QCNN/QAE), requiring a re-initialized center vector.

In [13]:
center = np.zeros(8)
center_train = np.tile(center, (len(X_train), 1))
lcqhnn_batch_size = 8
lcqhnn_steps = 1000
lcqhnn_learning_rate = 0.01
print(f'center_train shape: {center_train.shape}')
train_Xdata = X_train
train_Ydata = center_train

center_train shape: (600, 8)


#### Five-Run Training & Result Persistence

Run `train_five_times` to train the **QAE** ansatz across 5 independent runs with the configured batch size and step count.

In [14]:
(lcqhnn_loss_history_matrix,
 lcqhnn_est_params_matrix,
 lcqhnn_param_history_matrix,
 lcqhnn_time_record) = train_five_times(X_train=train_Xdata,
                                        Y_train=train_Ydata,
                                        batch_size=lcqhnn_batch_size,
                                        learning_rate=lcqhnn_learning_rate,
                                        steps=lcqhnn_steps,
                                        ansatz='lcqhnn'
                                        )
loss_history_f_name = f"../results/training/LCQHNN/MISC_LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}LR{lcqhnn_learning_rate:.0e}_LOSS_HISTORY_MEAN.npy"
est_params_f_name = f"../results/training/LCQHNN/MISC_LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}LR{lcqhnn_learning_rate:.0e}_EST_PARAMS_MEAN.npy"
time_f_name = f"../results/training/LCQHNN/MISC_LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}LR{lcqhnn_learning_rate:.0e}_TIME_MEAN.npy"
np.savetxt(loss_history_f_name, lcqhnn_loss_history_matrix)
np.savetxt(est_params_f_name, lcqhnn_est_params_matrix)
np.savetxt(time_f_name, lcqhnn_time_record)
print("--- All training batches completed ---")

--- Starting training round 1 with seed 1497434561 ---
--- Starting training round 2 with seed 3985803128 ---


KeyboardInterrupt: 

#### LCQHNN Training Evaluation

In [48]:
f_name = f"../results/training/LCQHNN/MISC_LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}LR{lcqhnn_learning_rate:.0e}_EST_PARAMS_MEAN.npy"
params_list = np.loadtxt(f_name)
mean, std = mean_auc(params_list, n_train, X_test, Y_test, center_train, noisy=False, ansatz="lcqhnn")

print(50*"--")
print(f'for: B{lcqhnn_batch_size}S{lcqhnn_steps} | AUC_mean: {mean} | std: {std}')
print(50*"--")

Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.21s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.21s
Test completed in 0.42s | AUC: 0.9373
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.20s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.20s
Test completed in 0.41s | AUC: 0.9061
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.20s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.20s
Test completed in 0.40s | AUC: 0.9443
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.20s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.20s
Test completed in 0.40s | AUC: 0.8385
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.20s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.20s
Test completed in 0.40s | AUC: 0.8782
----------------------------------------------------------------------------------------------------
for: B8S1

## Noisy (NISQ-Era) Training

In the noisy setting, the quantum circuits are simulated with a **hardware noise model** that mimics real NISQ (Noisy Intermediate-Scale Quantum) device behavior, including gate errors and decoherence. This evaluates model robustness under realistic quantum hardware conditions.

The same three ansatzes are retrained from scratch under this noisy simulation.